In [8]:
!pip install langchain
!pip install langchain-community
!pip install sentence-transformers
!pip install faiss-cpu
!pip install pypdf
!pip install -q langchain-text-splitters
!pip install -q sentence-transformers

In [3]:
!pip install -q langchain langchain-community sentence-transformers faiss-cpu pypdf google-generativeai

In [26]:
from google.colab import files

uploaded = files.upload()

Saving Resume.pdf to Resume.pdf


In [27]:
from langchain_community.document_loaders import PyPDFLoader

pdf_name = list(uploaded.keys())[0]

loader = PyPDFLoader(pdf_name)
documents = loader.load()

print("Total Pages:", len(documents))
print("\nFirst Page:\n")
print(documents[0].page_content)

Total Pages: 1

First Page:

SATYAM KUMAR
satyamkumar72240@gmail.com
 
+91 9263903007
 
Bhubaneswar, Odisha
 
linkedin.com/in/satyamkumar
 
github.com/SATYAM-KUMAR722
 
Objective
Computer Science student with strong skills in Full-Stack Development, Artificial Intelligence, Machine Learning, and 
Data Structures & Algorithms. Hands-on experience in building AI-powered and scalable web applications using 
technologies like React.js, Node.js, MongoDB, Python, Deep Learning, and NLP. Experienced in developing real-world 
projects that combine software engineering with intelligent systems and data-driven solutions. Passionate about 
problem-solving, modern technologies, and building impactful applications in dynamic environments.
Education
B.Tech In Computer Science and Engineering
Siksha 'O' Anusandhan University - CGPA: 8.69
09/2023 – 06/2027 | Bhubaneswar, Odisha
Intermediate, Guru Gobind Singh Public School 06/2023 | Bokaro Steel City, Jharkhand
•CBSE - 80%
Matriculation, Guru Gobind S

In [28]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

splitter = RecursiveCharacterTextSplitter(
    chunk_size=300,
    chunk_overlap=50
)

chunks = splitter.split_documents(documents)

print("Total Chunks:", len(chunks))
print(chunks[0].page_content)

Total Chunks: 13
SATYAM KUMAR
satyamkumar72240@gmail.com
 
+91 9263903007
 
Bhubaneswar, Odisha
 
linkedin.com/in/satyamkumar
 
github.com/SATYAM-KUMAR722
 
Objective
Computer Science student with strong skills in Full-Stack Development, Artificial Intelligence, Machine Learning, and


In [29]:
from sentence_transformers import SentenceTransformer

embedding_model = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")
texts = [chunk.page_content for chunk in chunks]
embeddings = embedding_model.encode(texts)

print("Number of Chunks:", len(texts))
print("Embedding Shape:", embeddings.shape)
print("Dimension of One Embedding:", len(embeddings[0]))

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Number of Chunks: 13
Embedding Shape: (13, 384)
Dimension of One Embedding: 384


In [30]:
from langchain_community.vectorstores import FAISS
from langchain_community.embeddings import HuggingFaceEmbeddings

embedding_model = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)

db = FAISS.from_documents(chunks, embedding_model)

db.save_local("vector_db")

print("Vector database created successfully!")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Vector database created successfully!


In [48]:
import google.generativeai as genai
from getpass import getpass

api_key = getpass("Enter your Gemini API key: ").strip()
genai.configure(api_key=api_key)

model = genai.GenerativeModel("gemini-2.5-flash")

Enter your Gemini API key: ··········


In [49]:
questions = [
    "What is the candidate's CGPA?",
    "Which university is the candidate studying at?",
    "What projects has the candidate built?",
    "Which internship is currently ongoing?"
]

for question in questions:
    print("=" * 80)
    print("Question:", question)

    results = db.similarity_search(question, k=3)
    context = "\n\n".join([doc.page_content for doc in results])

    print("\nRetrieved Context:\n")
    for i, doc in enumerate(results, 1):
        print(f"Chunk {i}:\n{doc.page_content}\n")

    prompt = f"""
You are a helpful assistant.

Answer ONLY using the provided context.
If the answer is not present in the context, reply:
'I couldn't find the answer in the provided document.'

Context:
{context}

Question:
{question}

Answer:
"""
    response = model.generate_content(prompt)
    print("Final Answer:\n", response.text)

Question: What is the candidate's CGPA?

Retrieved Context:

Chunk 1:
B.Tech In Computer Science and Engineering
Siksha 'O' Anusandhan University - CGPA: 8.69
09/2023 – 06/2027 | Bhubaneswar, Odisha
Intermediate, Guru Gobind Singh Public School 06/2023 | Bokaro Steel City, Jharkhand
•CBSE - 80%

Chunk 2:
SATYAM KUMAR
satyamkumar72240@gmail.com
 
+91 9263903007
 
Bhubaneswar, Odisha
 
linkedin.com/in/satyamkumar
 
github.com/SATYAM-KUMAR722
 
Objective
Computer Science student with strong skills in Full-Stack Development, Artificial Intelligence, Machine Learning, and

Chunk 3:
•CBSE - 80%
Matriculation, Guru Gobind Singh Public School 06/2021 | Bokaro Steel City, Jharkhand
•CBSE - 74.4%
Experience
Data Science Intern | Celebal Technologies 06/2026 – Present
•Design Python-based machine learning models to analyze complex datasets and optimize processes.

Final Answer:
 CGPA: 8.69
Question: Which university is the candidate studying at?

Retrieved Context:

Chunk 1:
B.Tech In Computer Sc

In [50]:
print("===== System Metrics =====")
print(f"Documents: 1")
print(f"Pages: {len(documents)}")
print(f"Chunks: {len(chunks)}")
print("Chunk Size: 300")
print("Chunk Overlap: 50")
print("Embedding Model: all-MiniLM-L6-v2")
print(f"Embedding Dimension: {embeddings.shape[1]}")
print("Vector Store: FAISS")
print("Retriever: Similarity Search")
print("LLM: Gemini 2.5 Flash")

===== System Metrics =====
Documents: 1
Pages: 1
Chunks: 13
Chunk Size: 300
Chunk Overlap: 50
Embedding Model: all-MiniLM-L6-v2
Embedding Dimension: 384
Vector Store: FAISS
Retriever: Similarity Search
LLM: Gemini 2.5 Flash
